# AI4BCI: second-architecture readout replication

Standalone notebook: upload to Colab, choose a GPU runtime, and Run all.
Local Jupyter also works. Google Drive persists checkpoints in Colab.

This post-study exploratory experiment tests whether source-only ridge refitting
recovers performance after smooth-to-spiking conversion in a structurally different
encoder. It uses the same BCI IV-2a dataset and nine participants, not independent data.
It does not repeat the SAST comparison.

**Default:** nine participants × (four tuning fits + five main fits) = 81 fits.
All source training and readout selection finish before target sessions are loaded.
Rerun unchanged after a disconnect to resume from completed epochs. Keep one notebook
runtime per output directory. Numerical results can differ across software/devices.

In [ ]:
import importlib.metadata, importlib.util, os, subprocess, sys, tempfile, hashlib, json, shutil
from pathlib import Path

required = ["numpy", "scipy", "pandas", "matplotlib", "torch"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

RUN_FULL_REPLICATION = True
USE_GOOGLE_DRIVE = True
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/AI4BCI_readout_replication")
else:
    PROJECT = Path(os.environ.get("AI4BCI_REPLICATION_DIR", "ai4bci_replication")).resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
print("Persistent outputs:", PROJECT)

## Fixed experiment

Original preprocessing: run-wise 4–38 Hz zero-phase filter, 0–4 s after the cue,
250→125 Hz resampling, source-training channel standardization. Session 1 runs 1–5
train (240 trials), run 6 validates (48), all session 2 tests (288). Artifact-marked
trials are retained, with artifact-free scores also exported. This is offline decoding.

Second encoder: 8 shared temporal filters (63 samples), depthwise electrode filters
(two per temporal filter), average pooling by 8, GroupNorm(1), one LIF block
(16 channels, leak 0.9, subtractive previous-activity reset, threshold 1), four
temporal-bin averages, linear 64→4 head. Linear spatial filtering is evaluated
before temporal filtering to reduce intermediates; a reference-value/gradient
check verifies that the filter ordering represents the same operation.

Ordinary smooth AdamW training only: 800 updates, batch 32, cosine LR schedule;
LR {0.0003, 0.001} × weight decay {0.0001, 0.001}. Pilot seed 1000 selects settings;
main seeds 0–4 use the selected source settings. Best checkpoint at 20 validation
points uses accuracy, then loss, then earlier epoch. No target-dependent selection.

Smooth and raw-spike final features receive separately fitted ridge heads with
the original seven-value source-validation penalty grid. Encoders stay frozen.
Four outcomes: smooth/original, smooth/ridge, raw/original, raw/ridge. Seed means
within participants precede paired 10,000-resample participant bootstrap intervals.
The smooth-refit control distinguishes general refitting gains from extra raw-spike
recovery. All results, including weak or contrary replication, must be retained.

In [ ]:
SOURCES = {'ai4bci_study.py': '"""Self-contained AI4BCI experiment engine, embedded in the deliverable notebook."""\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport itertools\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport sys\nimport time\nimport urllib.request\nimport zipfile\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport scipy\nfrom scipy.io import loadmat\nfrom scipy.signal import butter, resample_poly, sosfiltfilt\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nMETHODS = ("ordinary", "matched", "sam", "sast", "direct", "eegnet")\nLABELS = dict(zip(METHODS, ("Ordinary smooth", "Ordinary, 2x updates", "Same-batch SAM",\n                          "Independent SAST", "Direct hard", "EEGNet")))\nCHANNELS = "Fz FC3 FC1 FCz FC2 FC4 C5 C3 C1 Cz C2 C4 C6 CP3 CP1 CPz CP2 CP4 P1 Pz P2 POz".split()\nCLASS_NAMES = ["left hand", "right hand", "feet", "tongue"]\nENGINE_HASH = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\n\n@dataclass(frozen=True)\nclass BCIConfig:\n    subjects: tuple = tuple(range(1, 10))\n    seeds: tuple = tuple(range(5))\n    epochs: int = 100\n    batch_size: int = 32\n    widths: tuple = (16, 32)\n    leak: float = .9\n    samples: int = 500\n    learning_rates: tuple = (.0003, .001)\n    radii: tuple = (.05, .2)\n    weight_decays: tuple = (.0001, .001)\n    thresholds: tuple = (.25, .5, .75, 1., 1.25, 1.5, 2., 3.)\n    ridge_alphas: tuple = (.001, .01, .1, 1., 10., 100., 1000.)\n    budgets: tuple = (0, 4, 8, 12)\n    pilot_seed: int = 1000\n    bootstrap: int = 10000\n    synthetic: bool = False\n\n    def validate(self):\n        assert self.subjects and len(set(self.subjects)) == len(self.subjects)\n        assert set(self.subjects) <= set(range(1, 10))\n        assert self.seeds and len(set(self.seeds)) == len(self.seeds)\n        assert self.pilot_seed not in self.seeds\n        assert self.epochs > 0 and self.batch_size > 0 and self.samples >= 64\n        assert len(self.widths) == 2 and min(self.widths) > 0 and 0 < self.leak < 1\n        assert len(self.radii) == len(self.weight_decays) and len(self.radii) > 0\n        assert all(v > 0 for seq in (self.learning_rates, self.radii, self.weight_decays,\n                                     self.thresholds, self.ridge_alphas) for v in seq)\n        assert len(set(self.thresholds)) == len(self.thresholds) and 1. in self.thresholds\n        assert self.budgets[0] == 0 and tuple(sorted(set(self.budgets))) == self.budgets\n        assert self.bootstrap >= 100\n        if not self.synthetic:\n            assert self.samples == 500 and self.budgets == (0, 4, 8, 12)\n        return self\n\n\ndef atomic_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False))\n    os.replace(tmp, path)\n\n\ndef atomic_torch(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".tmp")\n    torch.save(value, tmp); os.replace(tmp, path)\n\n\ndef sha_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""): h.update(chunk)\n    return h.hexdigest()\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef select_device():\n    # CPU fallback is portable; CUDA is the intended full-study execution target.\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef environment():\n    return {"python": platform.python_version(), "torch": torch.__version__,\n            "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,\n            "matplotlib": matplotlib.__version__}\n\n\ndef prepare_root(cfg, output):\n    cfg.validate()\n    identity = {"config": asdict(cfg), "engine": ENGINE_HASH, "software": environment()}\n    name = ("SMOKE_" if cfg.synthetic else "BNCI2014_001_") + digest(identity)[:16]\n    root = Path(output) / name; root.mkdir(parents=True, exist_ok=True)\n    atomic_json(root / "study.json", identity)\n    # Store the implementation itself so the output can be audited independently.\n    (root / "ai4bci_study.py").write_text(Path(__file__).read_text())\n    return root\n\n\n# Data: the MATLAB trial markers precede the cue by TWO seconds.\ndef download_session(subject, session, cache):\n    name = f"A{subject:02d}{session}.mat"\n    path = Path(cache) / name; path.parent.mkdir(parents=True, exist_ok=True)\n    record = path.with_suffix(".sha256.json")\n    url = "https://lampx.tugraz.at/~bci/database/001-2014/" + name\n    if path.exists() and record.exists():\n        info = json.loads(record.read_text())\n        if sha_file(path) != info["sha256"]:\n            raise RuntimeError(f"Corrupt cached download: {path}; move it aside and rerun.")\n        return path\n    if not path.exists():\n        for attempt in range(3):\n            tmp = path.with_suffix(".part")\n            try:\n                request = urllib.request.Request(url, headers={"User-Agent": "AI4BCI-reproduction/1"})\n                with urllib.request.urlopen(request, timeout=90) as response, tmp.open("wb") as out:\n                    expected = response.headers.get("Content-Length")\n                    shutil.copyfileobj(response, out, length=1024 * 1024)\n                if expected and tmp.stat().st_size != int(expected):\n                    raise IOError("Incomplete dataset download")\n                loadmat(tmp, variable_names=["data"])\n                os.replace(tmp, path); break\n            except Exception:\n                if attempt == 2: raise\n                time.sleep(2 ** attempt)\n    loadmat(path, variable_names=["data"])\n    atomic_json(record, {"url": url, "sha256": sha_file(path), "bytes": path.stat().st_size})\n    return path\n\n\ndef read_session(path, subject, session):\n    runs = np.atleast_1d(loadmat(path, struct_as_record=False, squeeze_me=True)["data"])\n    arrays, rows = [], []\n    sos = butter(4, [4, 38], btype="bandpass", fs=250, output="sos")\n    run_id = 0\n    for raw_id, run in enumerate(runs):\n        markers = np.atleast_1d(run.trial).astype(int)\n        if not markers.size: continue  # Initial EOG/baseline recordings are not task runs.\n        run_id += 1\n        labels = np.atleast_1d(run.y).astype(int) - 1\n        flags = np.atleast_1d(run.artifacts).astype(int)\n        x = np.asarray(run.X, dtype=np.float64)\n        if float(run.fs) != 250 or x.ndim != 2 or x.shape[1] != 25:\n            raise ValueError("Unexpected sampling rate or channel count")\n        if len(markers) != 48 or len(flags) != 48 or not np.array_equal(np.bincount(labels, minlength=4), [12]*4):\n            raise ValueError("Unexpected trial/class/artifact counts; do not silently drop trials")\n        if not np.isfinite(x[:, :22]).all():\n            raise ValueError(f"Nonfinite EEG in {path}, task run {run_id}; inspect before analysis")\n        filtered = sosfiltfilt(sos, x[:, :22], axis=0, padtype="odd")\n        for trial, (marker, label, artifact) in enumerate(zip(markers, labels, flags)):\n            start = int(marker) - 1 + 500  # MATLAB one-based trial start + 2 s to cue.\n            epoch = filtered[start:start + 1000]\n            if epoch.shape != (1000, 22): raise ValueError("Incomplete 0--4 s cue epoch")\n            epoch = resample_poly(epoch, 1, 2, axis=0, window=("kaiser", 5.), padtype="constant")\n            arrays.append(epoch.T.astype(np.float32))\n            rows.append(dict(subject=subject, session=session, run=run_id, raw_run=raw_id,\n                             trial=trial, trial_start_sample=int(marker)-1, cue_sample=start,\n                             label=int(label), artifact=int(artifact),\n                             trial_id=f"A{subject:02d}{session}-r{run_id}-t{trial:02d}"))\n    if run_id != 6: raise ValueError(f"Expected six task runs; found {run_id}")\n    return np.stack(arrays), pd.DataFrame(rows)\n\n\ndef synthetic_session(cfg, subject, session):\n    rng = np.random.default_rng(9000 + subject * 2 + (session == "E"))\n    rows, xs = [], []\n    for run in range(1, 7):\n        labels = np.repeat(np.arange(4), 2); rng.shuffle(labels)\n        for j, label in enumerate(labels):\n            x = rng.normal(0, .2, (22, cfg.samples))\n            x[label*4:(label+1)*4] += np.sin(np.arange(cfg.samples) * (.1 + label*.03))\n            xs.append(x.astype(np.float32))\n            rows.append(dict(subject=subject, session=session, run=run, raw_run=run-1, trial=j,\n                             trial_start_sample=j*2000, cue_sample=j*2000+500, label=int(label),\n                             artifact=int(j == 0), trial_id=f"SYN-{subject}{session}-r{run}-t{j}"))\n    return np.stack(xs), pd.DataFrame(rows)\n\n\ndef prepare_subject(cfg, subject, cache, root, target=False):\n    directory = root / f"subject_{subject:02d}"; directory.mkdir(exist_ok=True)\n    def session(letter):\n        if cfg.synthetic: return (*synthetic_session(cfg, subject, letter), "synthetic")\n        path = download_session(subject, letter, cache)\n        checksum = sha_file(path)\n        prepared = directory / f"epochs_{letter}.npz"\n        meta = directory / f"trials_{letter}.csv"\n        if prepared.exists() and meta.exists():\n            with np.load(prepared, allow_pickle=False) as z:\n                if str(z["raw_sha"]) != checksum: raise RuntimeError("Dataset changed within a study")\n                return z["x"], pd.read_csv(meta), checksum\n        x, frame = read_session(path, subject, letter)\n        with prepared.with_suffix(".tmp").open("wb") as f: np.savez_compressed(f, x=x, raw_sha=checksum)\n        os.replace(prepared.with_suffix(".tmp"), prepared)\n        frame.to_csv(meta, index=False)\n        return x, frame, checksum\n    x, frame, checksum = session("T")\n    train = frame.run.to_numpy() <= 5\n    mean = x[train].mean(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)\n    scale = x[train].std(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)\n    scale = np.maximum(scale, 1e-6)\n    x = ((x - mean) / scale).astype(np.float32)\n    np.savez(directory / "source_scaler.npz", mean=mean, scale=scale)\n    frame.to_csv(directory / "trials_T.csv", index=False)\n    data = {"train": (x[train], frame[train].reset_index(drop=True)),\n            "val": (x[~train], frame[~train].reset_index(drop=True)), "source_sha": checksum}\n    if target:\n        xt, ft, target_sha = session("E")\n        ft.to_csv(directory / "trials_E.csv", index=False)\n        data["test"] = (((xt-mean)/scale).astype(np.float32), ft)\n        data["target_sha"] = target_sha\n    return data\n\n\n# Network definitions preserve the existing arctan and subtractive-reset semantics.\n@torch.jit.script\ndef lif_sequence(drive: torch.Tensor, leak: float, threshold: float, mode: int):\n    state = torch.zeros_like(drive[:, :, 0]); previous = torch.zeros_like(state)\n    outputs = torch.jit.annotate(list[torch.Tensor], [])\n    for t in range(drive.size(2)):\n        state = leak * state + drive[:, :, t] - threshold * previous\n        distance = state - threshold\n        smooth = .5 + torch.atan(distance) / math.pi\n        if mode == 0:\n            previous = smooth\n        else:\n            hard = (distance >= 0).to(drive.dtype)\n            previous = hard + (smooth - smooth.detach()) if mode == 2 else hard\n        outputs.append(previous)\n    return torch.stack(outputs, dim=2)\n\n\nclass EEGSpiking(nn.Module):\n    def __init__(self, cfg):\n        super().__init__(); a, b = cfg.widths; self.leak = cfg.leak\n        self.temporal = nn.Conv2d(1, 8, (1, 31), padding=(0, 15), bias=False)\n        self.spatial = nn.Conv2d(8, a, (22, 1), bias=True)\n        self.norm1 = nn.GroupNorm(1, a)\n        self.conv2 = nn.Conv1d(a, b, 5, padding=2)\n        self.norm2 = nn.GroupNorm(1, b)\n        self.head = nn.Linear(b, 4)\n\n    def forward(self, x, mode=0, threshold=1., features=False):\n        drive = self.spatial(self.temporal(x.unsqueeze(1))).squeeze(2)\n        drive = self.norm1(F.avg_pool1d(drive, 4))\n        z1 = lif_sequence(drive, self.leak, threshold, mode)\n        z2 = lif_sequence(self.norm2(self.conv2(z1)), self.leak, threshold, mode)\n        f1, f2 = z1.mean(2), z2.mean(2)\n        logits = self.head(f2)\n        return (logits, [f1, f2]) if features else logits\n\n\nclass EEGNet(nn.Module):\n    """PyTorch EEGNet-8,2 reference: 32-sample MI kernel, 125 Hz, 500 samples.\n\n    Architectural specification: Lawhern et al., EEGModels.py (BSD-3-Clause).\n    This independently written port keeps BN/ELU/dropout/pooling/max-norm;\n    uses PyTorch initialization and the common study optimizer/search.\n    """\n    def __init__(self, cfg):\n        super().__init__()\n        self.temporal = nn.Conv2d(1, 8, (1, 32), bias=False)\n        self.bn1 = nn.BatchNorm2d(8, eps=.001, momentum=.01)\n        self.spatial = nn.Conv2d(8, 16, (22, 1), groups=8, bias=False)\n        self.bn2 = nn.BatchNorm2d(16, eps=.001, momentum=.01)\n        self.depth = nn.Conv2d(16, 16, (1, 16), groups=16, bias=False)\n        self.point = nn.Conv2d(16, 16, 1, bias=False)\n        self.bn3 = nn.BatchNorm2d(16, eps=.001, momentum=.01)\n        self.dropout = nn.Dropout(.5)\n        self.head = nn.Linear(16 * (cfg.samples // 32), 4)\n\n    def forward(self, x, mode=0, threshold=1., features=False):\n        z = self.bn1(self.temporal(F.pad(x.unsqueeze(1), (15, 16))))\n        z = self.dropout(F.avg_pool2d(F.elu(self.bn2(self.spatial(z))), (1, 4)))\n        f1 = z.mean((2, 3))\n        z = self.point(self.depth(F.pad(z, (7, 8))))\n        z = self.dropout(F.avg_pool2d(F.elu(self.bn3(z)), (1, 8)))\n        f2 = z.flatten(1); logits = self.head(f2)\n        return (logits, [f1, f2]) if features else logits\n\n    @torch.no_grad()\n    def constrain(self):\n        for p, limit in ((self.spatial.weight, 1.), (self.head.weight, .25)):\n            flat = p.flatten(1); norms = flat.norm(dim=1, keepdim=True).clamp_min(1e-12)\n            p.mul_((limit / norms).clamp(max=1).reshape(-1, *([1]*(p.ndim-1))))\n\n\ndef make_model(cfg, method, subject, seed, device):\n    seed_all(subject * 10000 + seed)\n    return (EEGNet(cfg) if method == "eegnet" else EEGSpiking(cfg)).to(device)\n\n\ndef balanced_accuracy(y, pred):\n    y, pred = np.asarray(y), np.asarray(pred)\n    if not all(np.any(y == c) for c in range(4)): return float("nan")\n    return float(np.mean([np.mean(pred[y == c] == c) for c in range(4)]) * 100)\n\n\n@torch.no_grad()\ndef infer(model, x, device, mode=0, threshold=1., batch=64):\n    model.eval(); logits, feats = [], [[], []]\n    for offset in range(0, len(x), batch):\n        tensor = torch.as_tensor(x[offset:offset+batch], device=device)\n        out, fs = model(tensor, mode, float(threshold), features=True)\n        logits.append(out.cpu().numpy())\n        for target, values in zip(feats, fs): target.append(values.cpu().numpy())\n    result = np.concatenate(logits)\n    if not np.isfinite(result).all(): raise FloatingPointError("Nonfinite evaluation output")\n    return result, [np.concatenate(v) for v in feats]\n\n\ndef training_step(model, optimizer, xd, yd, xa=None, ya=None, rho=0., hard=False):\n    model.train(); optimizer.zero_grad(set_to_none=True)\n    mode = 2 if hard else 0\n    def backward(x, y):\n        loss = F.cross_entropy(model(x, mode), y)\n        if not torch.isfinite(loss): raise FloatingPointError("Nonfinite training loss")\n        loss.backward()\n        norm = torch.linalg.vector_norm(torch.stack([p.grad.norm() for p in model.parameters() if p.grad is not None]))\n        if not torch.isfinite(norm): raise FloatingPointError("Nonfinite gradient")\n        return loss, norm\n    if rho:\n        _, norm = backward(xa, ya)\n        originals = [(p, p.detach().clone()) for p in model.parameters()]\n        try:\n            with torch.no_grad():\n                for p, _ in originals:\n                    if p.grad is not None: p.add_(p.grad * (rho/(norm+1e-12)))\n            optimizer.zero_grad(set_to_none=True)\n            loss, _ = backward(xd, yd)\n        finally:\n            with torch.no_grad():\n                for p, old in originals: p.copy_(old)\n    else:\n        loss, _ = backward(xd, yd)\n    optimizer.step()\n    if isinstance(model, EEGNet): model.constrain()\n    return float(loss.detach())\n\n\ndef candidate_settings(cfg, method):\n    if method in ("sam", "sast"):\n        return [dict(lr=lr, rho=r, wd=.0001) for lr in cfg.learning_rates for r in cfg.radii]\n    return [dict(lr=lr, rho=0., wd=w) for lr in cfg.learning_rates for w in cfg.weight_decays]\n\n\ndef fit_model(cfg, method, subject, seed, settings, data, directory, device, stop_after=None):\n    directory.mkdir(parents=True, exist_ok=True)\n    epochs = cfg.epochs * (2 if method == "matched" else 1)\n    identity = digest(dict(config=asdict(cfg), method=method, subject=subject, seed=seed,\n                           settings=settings, source=data["source_sha"], engine=ENGINE_HASH,\n                           software=environment(), device=device.type))\n    model = make_model(cfg, method, subject, seed, device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["lr"], weight_decay=settings["wd"])\n    descent = torch.Generator().manual_seed(subject*10000 + seed + 100000)\n    ascent = torch.Generator().manual_seed(subject*10000 + seed + 200000)\n    state_path = directory / "resume.pt"\n    start, history, best, elapsed = 0, [], None, 0.\n    if state_path.exists():\n        saved = torch.load(state_path, map_location="cpu", weights_only=False)\n        if saved["identity"] != identity: raise RuntimeError(f"Resume identity changed: {directory}")\n        model.load_state_dict(saved["model"]); optimizer.load_state_dict(saved["optimizer"])\n        start, history, best, elapsed = saved["epoch"], saved["history"], saved["best"], saved["seconds"]\n        descent.set_state(saved["descent"]); ascent.set_state(saved["ascent"])\n        torch.set_rng_state(saved["rng"])\n        if device.type == "cuda": torch.cuda.set_rng_state_all(saved["cuda_rng"])\n    xt, ft = data["train"]; xv, fv = data["val"]\n    x = torch.as_tensor(xt, device=device); y = torch.as_tensor(ft.label.to_numpy(), device=device)\n    steps = math.ceil(len(x)/cfg.batch_size)\n    validation_every = max(1, math.ceil(epochs/20))\n    for epoch in range(start, epochs):\n        tick = time.perf_counter(); losses = []\n        for step in range(steps):\n            progress = (epoch*steps+step)/max(1, epochs*steps-1)\n            lr = settings["lr"] * (.01 + .99 * (1+math.cos(math.pi*progress))/2)\n            for group in optimizer.param_groups: group["lr"] = lr\n            ids = torch.randint(len(x), (cfg.batch_size,), generator=descent).to(device)\n            aid = (torch.randint(len(x), (cfg.batch_size,), generator=ascent).to(device)\n                   if method == "sast" else ids)\n            losses.append(training_step(model, optimizer, x[ids], y[ids], x[aid], y[aid],\n                                        settings["rho"], hard=method == "direct"))\n        if device.type == "cuda": torch.cuda.synchronize()\n        entry = dict(epoch=epoch+1, train_loss=float(np.mean(losses)), validation_ba=None,\n                     validation_loss=None, gradient_evaluations=(epoch+1)*steps*(2 if settings["rho"] else 1))\n        if (epoch+1) % validation_every == 0 or epoch+1 == epochs:\n            logits, _ = infer(model, xv, device, mode=1 if method == "direct" else 0)\n            ba = balanced_accuracy(fv.label, logits.argmax(1))\n            vl = float(F.cross_entropy(torch.from_numpy(logits), torch.from_numpy(fv.label.to_numpy())))\n            entry.update(validation_ba=ba, validation_loss=vl)\n            if best is None or (ba, -vl) > (best["ba"], -best["loss"]):\n                best = dict(ba=ba, loss=vl, epoch=epoch+1,\n                            weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()})\n        history.append(entry)\n        elapsed += time.perf_counter()-tick  # Training plus validation; excludes checkpoint I/O.\n        atomic_torch(state_path, dict(identity=identity, epoch=epoch+1, history=history, best=best,\n                     seconds=elapsed, model={k:v.detach().cpu() for k,v in model.state_dict().items()},\n                     optimizer=optimizer.state_dict(), descent=descent.get_state(), ascent=ascent.get_state(),\n                     rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if device.type=="cuda" else []))\n        if (epoch+1) % max(1, epochs//5) == 0:\n            print(f"  {directory.name}: epoch {epoch+1}/{epochs}, loss {entry[\'train_loss\']:.4f}", flush=True)\n        if stop_after is not None and epoch+1 >= stop_after and epoch+1 < epochs:\n            return None\n    model.load_state_dict(best["weights"])\n    pd.DataFrame(history).to_csv(directory / "history.csv", index=False)\n    atomic_torch(directory / "best.pt", best["weights"])\n    atomic_json(directory / "training.json", dict(settings=settings, best_epoch=best["epoch"],\n                validation_ba=best["ba"], validation_loss=best["loss"], seconds=elapsed,\n                gradient_evaluations=history[-1]["gradient_evaluations"],\n                parameters=sum(p.numel() for p in model.parameters()), seed=seed, method=method))\n    return model, best, history, elapsed\n\n\ndef train_all(cfg, cache, root, device):\n    # Finish every validation-only search and main run before opening target sessions.\n    for subject in cfg.subjects:\n        data = prepare_subject(cfg, subject, cache, root)\n        for method in METHODS:\n            directory = root/f"subject_{subject:02d}"/method\n            print(f"SOURCE ONLY: participant {subject}, {method}", flush=True)\n            candidates = []\n            for j, settings in enumerate(candidate_settings(cfg, method)):\n                model, best, history, seconds = fit_model(cfg, method, subject, cfg.pilot_seed,\n                    settings, data, directory/f"pilot_{j}", device)\n                candidates.append(dict(candidate=j, settings=settings, validation_ba=best["ba"],\n                                       validation_loss=best["loss"], seconds=seconds))\n                del model\n            winner = min(candidates, key=lambda c:(-c["validation_ba"], c["validation_loss"], c["candidate"]))\n            atomic_json(directory/"selection.json", dict(winner=winner, candidates=candidates))\n            for seed in cfg.seeds:\n                print(f"MAIN: participant {subject}, {method}, seed {seed}", flush=True)\n                fit_model(cfg, method, subject, seed, winner["settings"], data,\n                          directory/f"seed_{seed}", device)\n    atomic_json(root/"training_complete.json", {"subjects":list(cfg.subjects), "seeds":list(cfg.seeds)})\n\n\ndef select_threshold(predictions, labels, grid, reference=1.):\n    return min(grid, key=lambda t:(-balanced_accuracy(labels, predictions[t]), abs(t-reference), t))\n\n\ndef linear_cka(x, y):\n    x = np.asarray(x, dtype=np.float64); y = np.asarray(y, dtype=np.float64)\n    x = x-x.mean(0); y = y-y.mean(0)\n    denominator = np.linalg.norm(x.T@x) * np.linalg.norm(y.T@y)\n    return float(np.linalg.norm(x.T@y)**2/denominator) if denominator > 1e-20 else float("nan")\n\n\ndef fit_ridge(x, y, alpha, scaler=None):\n    x = np.asarray(x, dtype=np.float64)\n    if scaler is None: scaler = (x.mean(0), np.maximum(x.std(0), 1e-6))\n    z = (x-scaler[0])/scaler[1]\n    zm = z.mean(0); targets = np.eye(4)[np.asarray(y, dtype=int)]; ym = targets.mean(0)\n    zc = z-zm\n    weights = np.linalg.solve(zc.T@zc + alpha*np.eye(z.shape[1]), zc.T@(targets-ym))\n    return dict(weights=weights, intercept=ym-zm@weights, mean=scaler[0], scale=scaler[1], alpha=alpha)\n\n\ndef ridge_predict(head, x):\n    return (((x-head["mean"])/head["scale"])@head["weights"]+head["intercept"]).argmax(1)\n\n\ndef select_ridge(xt, yt, xv, yv, grid):\n    candidates = [fit_ridge(xt, yt, a) for a in grid]\n    return max(candidates, key=lambda h:(balanced_accuracy(yv, ridge_predict(h, xv)), h["alpha"]))\n\n\ndef calibration_indices(frame, k):\n    pool = np.flatnonzero(frame.run.to_numpy() == 1)\n    ids = np.concatenate([pool[frame.label.to_numpy()[pool] == c][:k] for c in range(4)])\n    if len(ids) != 4*k: raise ValueError("Calibration pool has too few trials per class")\n    return np.sort(ids)\n\n\ndef evaluate_checkpoint(cfg, model, method, subject, seed, data, directory, device):\n    directory.mkdir(exist_ok=True)\n    xt, ft = data["train"]; xv, fv = data["val"]; xe, fe = data["test"]\n    yt, yv, ye = (f.label.to_numpy() for f in (ft, fv, fe))\n    modes, val_grid, test_grid = {}, {}, {}\n    if method != "eegnet":\n        for threshold in cfg.thresholds:\n            val_grid[threshold] = infer(model, xv, device, 1, threshold)[0].argmax(1)\n        chosen = select_threshold(val_grid, yv, cfg.thresholds)\n        modes.update(raw=(1, 1.), calibrated=(1, chosen))\n    else: chosen = None\n    if method != "direct": modes["continuous"] = (0, 1.)\n    atomic_json(directory/"calibration.json", dict(threshold=chosen,\n                validation_scores={str(t):balanced_accuracy(yv,p) for t,p in val_grid.items()}))\n    rows1, rows2, rows3, predictions, features, heads = [], [], [], [], {}, {}\n    def record(experiment, mode, branch, budget, pred, ids):\n        frame = fe.iloc[ids].copy()\n        frame["prediction"] = np.asarray(pred)[ids]; frame["correct"] = frame.prediction == frame.label\n        frame["experiment"] = experiment; frame["mode"] = mode\n        frame["intervention"] = branch; frame["budget"] = budget\n        frame["method"] = method; frame["seed"] = seed; predictions.append(frame)\n    def scores(pred, ids):\n        clean = ids[fe.artifact.to_numpy()[ids] == 0]\n        return dict(ba=balanced_accuracy(ye[ids], pred[ids]),\n                    artifact_free_ba=balanced_accuracy(ye[clean], pred[clean]),\n                    prediction_classes=int(len(np.unique(pred[ids]))))\n    all_ids = np.arange(len(fe)); test_ids = np.flatnonzero(fe.run.to_numpy() >= 2)\n    for mode, (flag, threshold) in modes.items():\n        out_t, f_t = infer(model, xt, device, flag, threshold)\n        out_v, f_v = infer(model, xv, device, flag, threshold)\n        out_e, f_e = infer(model, xe, device, flag, threshold)\n        pred = out_e.argmax(1); features[mode] = (f_t, f_v, f_e)\n        rows1.append(dict(subject=subject, seed=seed, method=method, mode=mode,\n                          threshold=threshold, **scores(pred, all_ids)))\n        record("E1", mode, "original", 0, pred, all_ids)\n        for layer in range(2):\n            head = select_ridge(f_t[layer], yt, f_v[layer], yv, cfg.ridge_alphas)\n            probe_pred = ridge_predict(head, f_e[layer]); heads[(mode, layer)] = head\n            rows2.append(dict(subject=subject, seed=seed, method=method, mode=mode,\n                layer=layer+1, alpha=head["alpha"], probe_ba=balanced_accuracy(ye,probe_pred),\n                original_ba=balanced_accuracy(ye,pred), recovery_pp=balanced_accuracy(ye,probe_pred)-balanced_accuracy(ye,pred),\n                artifact_free_probe_ba=balanced_accuracy(ye[fe.artifact==0],probe_pred[fe.artifact==0]), cka=np.nan))\n            record("E2", mode, f"probe_L{layer+1}", 0, probe_pred, all_ids)\n        if mode != "raw":\n            base = heads[(mode, 1)]\n            for k in cfg.budgets:\n                ids = calibration_indices(fe, k)\n                adapted = fit_ridge(np.concatenate([f_t[1], f_e[1][ids]]), np.concatenate([yt, ye[ids]]),\n                                    base["alpha"], (base["mean"],base["scale"]))\n                ap = ridge_predict(adapted, f_e[1])\n                rows3.append(dict(subject=subject, seed=seed, method=method, mode=mode,\n                    intervention="readout", budget=4*k, threshold=threshold, **scores(ap,test_ids)))\n                record("E3",mode,"readout",4*k,ap,test_ids)\n    if "continuous" in features and method != "eegnet":\n        for row in rows2:\n            row["cka"] = linear_cka(features["continuous"][2][row["layer"]-1],\n                                    features[row["mode"]][2][row["layer"]-1])\n    if chosen is not None:\n        for t in cfg.thresholds: test_grid[t] = infer(model, xe, device, 1, t)[0].argmax(1)\n        calibration_manifest = []\n        for k in cfg.budgets:\n            ids = calibration_indices(fe, k)\n            t = chosen if k == 0 else select_threshold({t:p[ids] for t,p in test_grid.items()},ye[ids],cfg.thresholds,chosen)\n            pred = test_grid[t]\n            rows3.append(dict(subject=subject, seed=seed, method=method, mode="calibrated",\n                intervention="threshold",budget=4*k,threshold=t,**scores(pred,test_ids)))\n            record("E3","calibrated","threshold",4*k,pred,test_ids)\n            calibration_manifest.append(dict(budget=4*k,trial_ids=fe.iloc[ids].trial_id.tolist()))\n        atomic_json(directory/"calibration_trials.json", calibration_manifest)\n    for name, rows in (("e1",rows1),("e2",rows2),("e3",rows3)):\n        pd.DataFrame(rows).to_csv(directory/(name+".csv"),index=False)\n    pd.concat(predictions, ignore_index=True).to_csv(directory/"predictions.csv.gz",index=False)\n    np.savez_compressed(directory/"features.npz", **{f"{mode}_{split}_L{layer+1}":values[layer]\n                        for mode, fs in features.items() for split,values in zip(("train","val","test"),fs) for layer in range(2)})\n    np.savez_compressed(directory/"probes.npz", **{f"{mode}_L{layer+1}_{key}":value\n                        for (mode,layer),head in heads.items() for key,value in head.items()})\n    atomic_json(directory/"complete.json", dict(source_sha=data["source_sha"], target_sha=data["target_sha"]))\n\n\ndef analyze_all(cfg, cache, root, device):\n    if not (root/"training_complete.json").exists(): raise RuntimeError("Complete source-only training first")\n    for subject in cfg.subjects:\n        data = prepare_subject(cfg, subject, cache, root, target=True)\n        for method in METHODS:\n            for seed in cfg.seeds:\n                run = root/f"subject_{subject:02d}"/method/f"seed_{seed}"\n                out = run/"analysis"\n                if (out/"complete.json").exists():\n                    old = json.loads((out/"complete.json").read_text())\n                    if old != dict(source_sha=data["source_sha"],target_sha=data["target_sha"]):\n                        raise RuntimeError("Data changed since analysis")\n                    continue\n                model = make_model(cfg, method, subject, seed, device)\n                model.load_state_dict(torch.load(run/"best.pt",map_location=device,weights_only=True))\n                print(f"E1/E2/E3: participant {subject}, {method}, seed {seed}",flush=True)\n                evaluate_checkpoint(cfg,model,method,subject,seed,data,out,device)\n\n\ndef interval(values, cfg):\n    values = np.asarray(values, dtype=float)\n    if not np.isfinite(values).all(): return (float("nan"),)*3\n    rng = np.random.default_rng(20261012)\n    means = values[rng.integers(len(values),size=(cfg.bootstrap,len(values)))].mean(1)\n    return float(values.mean()), *np.quantile(means,[.025,.975]).tolist()\n\n\ndef summarize(frame, groups, metrics, cfg):\n    # Participant is the sampling unit; average all configured seeds FIRST.\n    participant = frame.groupby(groups+["subject"],dropna=False)[metrics].agg(\n        lambda values: values.mean() if np.isfinite(values).all() else np.nan).reset_index()\n    rows=[]\n    for keys, part in participant.groupby(groups,dropna=False):\n        keys = keys if isinstance(keys,tuple) else (keys,)\n        for metric in metrics:\n            mean, low, high = interval(part[metric],cfg)\n            rows.append(dict(zip(groups,keys),metric=metric,mean=mean,low=low,high=high,\n                             n_subjects=len(part),n_finite=int(np.isfinite(part[metric]).sum())))\n    return participant,pd.DataFrame(rows)\n\n\ndef report_all(cfg, root):\n    reports=root/"reports";reports.mkdir(exist_ok=True)\n    parts={key:[] for key in ("e1","e2","e3")}; cost=[]\n    for subject,method,seed in itertools.product(cfg.subjects,METHODS,cfg.seeds):\n        run=root/f"subject_{subject:02d}"/method/f"seed_{seed}"\n        if not (run/"analysis/complete.json").exists(): raise RuntimeError(f"Missing result: {run}")\n        for key in parts: parts[key].append(pd.read_csv(run/"analysis"/(key+".csv")))\n        cost.append(dict(subject=subject,**json.loads((run/"training.json").read_text())))\n    frames={k:pd.concat(v,ignore_index=True) for k,v in parts.items()}\n    expected_e1=len(cfg.subjects)*len(cfg.seeds)*15\n    if len(frames["e1"]) != expected_e1: raise RuntimeError("Incomplete/duplicate E1 mode matrix")\n    if len(frames["e2"]) != 2*expected_e1 or len(frames["e3"]) != len(cfg.budgets)*expected_e1:\n        raise RuntimeError("Incomplete E2/E3 matrix; do not report a partial experiment")\n    for key, columns in {"e1":["subject","seed","method","mode"],\n                         "e2":["subject","seed","method","mode","layer"],\n                         "e3":["subject","seed","method","mode","intervention","budget"]}.items():\n        if frames[key].duplicated(columns).any(): raise RuntimeError(f"Duplicate {key} results")\n    definitions={"e1":(["method","mode"],["ba","artifact_free_ba"]),\n                 "e2":(["method","mode","layer"],["probe_ba","original_ba","recovery_pp","cka","artifact_free_probe_ba"]),\n                 "e3":(["method","mode","intervention","budget"],["ba","artifact_free_ba"])}\n    summaries={}\n    for key,frame in frames.items():\n        frame.to_csv(reports/(key+"_per_seed.csv"),index=False)\n        groups,metrics=definitions[key]\n        participants,summary=summarize(frame,groups,metrics,cfg)\n        participants.to_csv(reports/(key+"_participants.csv"),index=False)\n        summary.to_csv(reports/(key+"_summary.csv"),index=False);summaries[key]=summary\n    e1,e2,e3=(frames[k] for k in ("e1","e2","e3"))\n    wide=e1.query("mode == \'calibrated\'").pivot(index=["subject","seed"],columns="method",values="ba")\n    paired=(wide.sast-wide.matched).groupby(level="subject").mean()\n    mean,lo,hi=interval(paired.values,cfg)\n    signs=np.array(list(itertools.product([-1,1],repeat=len(paired))))\n    p=float(np.mean(np.abs((signs*paired.values).mean(1)) >= abs(mean)-1e-12))\n    primary=dict(effect_pp=mean,low=lo,high=hi,sign_flip_p=p,n_subjects=len(paired),\n                 seeds_per_subject=len(cfg.seeds),synthetic=cfg.synthetic)\n    clean_wide=e1.query("mode == \'calibrated\'").pivot(index=["subject","seed"],columns="method",values="artifact_free_ba")\n    clean_paired=(clean_wide.sast-clean_wide.matched).groupby(level="subject").agg(\n        lambda values:values.mean() if np.isfinite(values).all() else np.nan)\n    clean_result=interval(clean_paired.values,cfg)\n    primary["artifact_free_sensitivity"]={k:(v if np.isfinite(v) else None)\n        for k,v in zip(("effect_pp","low","high"),clean_result)}\n    clean_paired.rename("sast_minus_matched_pp").to_csv(reports/"artifact_free_primary_participant_effects.csv")\n    atomic_json(reports/"primary_result.json",primary)\n    paired.rename("sast_minus_matched_pp").to_csv(reports/"primary_participant_effects.csv")\n    conversion=e1.pivot(index=["subject","seed","method"],columns="mode",values="ba").reset_index()\n    conversion["smooth_minus_raw_pp"]=conversion.continuous-conversion.raw\n    conversion["smooth_minus_calibrated_pp"]=conversion.continuous-conversion.calibrated\n    conversion=conversion[conversion.method.isin(METHODS[:4])]\n    conversion.to_csv(reports/"conversion_gaps_per_seed.csv",index=False)\n    _,cg=summarize(conversion,["method"],["smooth_minus_raw_pp","smooth_minus_calibrated_pp"],cfg)\n    cg.to_csv(reports/"conversion_gaps_summary.csv",index=False)\n    baseline=e3.query("budget == 0")[["subject","seed","method","mode","intervention","ba"]].rename(columns={"ba":"zero_ba"})\n    gains=e3.merge(baseline,on=["subject","seed","method","mode","intervention"],validate="many_to_one")\n    gains["gain_pp"]=gains.ba-gains.zero_ba\n    gains.to_csv(reports/"adaptation_gains_per_seed.csv",index=False)\n    _,gs=summarize(gains,["method","mode","intervention","budget"],["gain_pp"],cfg)\n    gs.to_csv(reports/"adaptation_gains_summary.csv",index=False)\n    pd.DataFrame(cost).to_csv(reports/"training_cost.csv",index=False)\n    tuning=[]\n    for subject,method in itertools.product(cfg.subjects,METHODS):\n        selection=json.loads((root/f"subject_{subject:02d}"/method/"selection.json").read_text())\n        tuning.extend(dict(subject=subject,method=method,**c) for c in selection["candidates"])\n    pd.DataFrame(tuning).to_csv(reports/"tuning_cost.csv",index=False)\n    counts=e1.groupby(["method","mode"]).prediction_classes.agg(["min","max"])\n    counts["single_class_runs"]=e1.groupby(["method","mode"]).prediction_classes.apply(lambda s:int((s==1).sum()))\n    counts.to_csv(reports/"collapse_counts.csv")\n    # Confusion matrices come directly from saved trial-level predictions.\n    conf=[]\n    for subject,method,seed in itertools.product(cfg.subjects,METHODS,cfg.seeds):\n        file=root/f"subject_{subject:02d}"/method/f"seed_{seed}"/"analysis/predictions.csv.gz"\n        pr=pd.read_csv(file)\n        for keys,g in pr.groupby(["experiment","mode","intervention","budget"]):\n            matrix=np.zeros((4,4),dtype=int);np.add.at(matrix,(g.label,g.prediction),1)\n            for true,pred in itertools.product(range(4),repeat=2):\n                conf.append(dict(subject=subject,method=method,seed=seed,experiment=keys[0],mode=keys[1],\n                            intervention=keys[2],budget=keys[3],true=true,predicted=pred,count=matrix[true,pred]))\n    pd.DataFrame(conf).to_csv(reports/"confusion_counts.csv",index=False)\n    make_figures(cfg,frames,summaries,gs,reports)\n    def cell(method,mode):\n        row=summaries["e1"].query("method == @method and mode == @mode and metric == \'ba\'")\n        if row.empty:return "--"\n        r=row.iloc[0];return f"{r[\'mean\']:.2f} [{r.low:.2f}, {r.high:.2f}]"\n    lines=[r"\\begingroup\\small\\setlength{\\tabcolsep}{4pt}",r"\\begin{tabular}{lccc}",\n           r"\\toprule",r"Method & Continuous & Raw spikes & Calibrated spikes \\\\",r"\\midrule"]\n    for method in METHODS:\n        lines.append(LABELS[method]+" & "+" & ".join(cell(method,m) for m in ("continuous","raw","calibrated"))+r" \\\\")\n    lines += [r"\\bottomrule",r"\\end{tabular}\\endgroup"]\n    (reports/"table1.tex").write_text("\\n".join(lines)+"\\n")\n    full = not cfg.synthetic and cfg.subjects==tuple(range(1,10)) and cfg.seeds==tuple(range(5))\n    conclusion=("The interval excludes zero in favor of SAST." if lo>0 else\n                "The interval excludes zero in favor of matched ordinary training." if hi<0 else\n                "The interval includes zero; this does not establish an advantage or equivalence.")\n    text=(f"# {\'COMPLETE DATASET RUN\' if full else \'SMOKE OR REDUCED RUN — NOT THE FULL PAPER RESULT\'}\\n\\n"\n          f"Calibrated SAST minus calibrated compute-matched ordinary training: **{mean:+.2f} pp** "\n          f"(paired participant bootstrap 95% interval [{lo:+.2f}, {hi:+.2f}]); "\n          f"{len(paired)} participants, {len(cfg.seeds)} seeds each. {conclusion}\\n\\n"\n          f"Exact sign-flip sensitivity p={p:.4g}, conditional on sign exchangeability.\\n\\n"\n          "- Table 1: `table1.tex` and `e1_summary.csv` (accuracy and artifact-free checks).\\n"\n          "- E1 within-checkpoint conversion loss: `conversion_gaps_summary.csv`.\\n"\n          "- E2: `e2_summary.csv`; L2 calibrated `cka`, `probe_ba`, and `recovery_pp` fill the draft.\\n"\n          "- E3: `e3_summary.csv` and `adaptation_gains_summary.csv`; budget 48 is the final comparison.\\n"\n          "- Main two-panel figure: `draft_results.pdf`; all curves: `adaptation_all.pdf`.\\n"\n          "- Training/search costs and collapsed runs have separate CSVs.\\n\\n"\n          "Do not infer mechanism from CKA alone. Review absolute decoding quality and direct-hard/EEGNet controls. "\n          "Ethics/data-use wording, manuscript status, public artifact URL, and the final interpretation require author review. "\n          "Raw EEG is not included in the results ZIP. The notebook does not submit or publish anything.\\n")\n    (reports/"RESULTS_FOR_DRAFT.md").write_text(text)\n    atomic_json(root/"complete.json",dict(full_protocol=full,primary=primary))\n    return reports\n\n\ndef make_figures(cfg,frames,summaries,gains,reports):\n    fig,axes=plt.subplots(1,2,figsize=(10,3.5),layout="constrained")\n    e2=frames["e2"].query("layer == 2 and mode == \'calibrated\'")\n    for j,method in enumerate(METHODS[:4]):\n        part=e2[e2.method==method].groupby("subject").recovery_pp.mean()\n        mean,lo,hi=interval(part,cfg)\n        axes[0].scatter(np.full(len(part),j),part,alpha=.4,s=15)\n        axes[0].errorbar(j,mean,yerr=[[mean-lo],[hi-mean]],fmt="ko",capsize=3)\n    axes[0].set_xticks(range(4),["Ordinary","2x updates","SAM","SAST"],rotation=15)\n    axes[0].axhline(0,color="gray",lw=.8);axes[0].set_ylabel("Spike probe − original readout (pp)")\n    axes[0].set_title("E2: Readout recovery")\n    es=summaries["e3"].query("metric == \'ba\'")\n    for method,mode in [(m,"calibrated") for m in METHODS[:5]]+[("eegnet","continuous")]:\n        g=es.query("method == @method and mode == @mode and intervention == \'readout\'").sort_values("budget")\n        axes[1].plot(g.budget,g["mean"],marker="o",label=LABELS[method])\n        axes[1].fill_between(g.budget,g.low,g.high,alpha=.09)\n    axes[1].set(xlabel="Labeled target trials",ylabel="Balanced accuracy (%)",title="E3: Readout adaptation")\n    axes[1].legend(fontsize=7,ncol=2)\n    if cfg.synthetic:fig.suptitle("SYNTHETIC PIPELINE CHECK — NOT EEG RESULTS",color="red")\n    fig.savefig(reports/"draft_results.pdf");fig.savefig(reports/"draft_results.png",dpi=200);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4),layout="constrained")\n    for ax,branch in zip(axes,("threshold","readout")):\n        for (method,mode),g in es[es.intervention==branch].groupby(["method","mode"]):\n            g=g.sort_values("budget");ax.plot(g.budget,g["mean"],marker="o",label=LABELS[method]+" / "+mode)\n        ax.set(title=branch+" adaptation",xlabel="Labeled target trials",ylabel="Balanced accuracy (%)")\n        ax.legend(fontsize=6)\n    fig.savefig(reports/"adaptation_all.pdf");plt.close(fig)\n\n\ndef export_zip(root):\n    destination=root.parent/(root.name+"_results.zip")\n    temp=destination.with_suffix(".tmp")\n    with zipfile.ZipFile(temp,"w",compression=zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(root.rglob("*")):\n            # Resume files and cached epochs can be large; best.pt and derived outputs are enough to audit.\n            if path.is_file() and path.name != "resume.pt" and not path.name.startswith("epochs_") and path.suffix not in (".tmp",".part"):\n                z.write(path,path.relative_to(root))\n    os.replace(temp,destination)\n    return destination\n\n\ndef run_study(cfg,cache,output,device=None):\n    device=torch.device(device) if device else select_device()\n    torch.set_num_threads(min(4,os.cpu_count() or 1))\n    torch.use_deterministic_algorithms(True)\n    if device.type=="cuda":\n        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n        torch.backends.cudnn.benchmark=False\n    root=prepare_root(cfg,output)\n    atomic_json(root/f"session_{time.time_ns()}.json",dict(software=environment(),device=str(device),\n        hardware=torch.cuda.get_device_name(device) if device.type=="cuda" else platform.processor()))\n    print("Results:",root,flush=True)\n    stage="training"\n    try:\n        train_all(cfg,cache,root,device)\n        stage="analysis";analyze_all(cfg,cache,root,device)\n        stage="reporting";report_all(cfg,root);archive=export_zip(root)\n    except BaseException as error:\n        atomic_json(root/f"failure_{time.time_ns()}.json",\n                    dict(stage=stage,type=type(error).__name__,message=str(error)))\n        raise\n    print("Completed:",archive,flush=True)\n    return root,archive\n\n\ndef smoke_config():\n    return BCIConfig(subjects=(1,2),seeds=(0,),epochs=1,batch_size=16,widths=(2,4),samples=128,\n        learning_rates=(.001,),radii=(.1,),weight_decays=(.0001,),thresholds=(.5,1.,2.),\n        ridge_alphas=(.1,1.),budgets=(0,1,2),bootstrap=200,synthetic=True)\n', 'replicate_readout.py': '"""Post-study, second-architecture readout replication; source-only selection.\n\nRun from the release root: python code/replicate_readout.py --output replication/runs\nThe default is nine participants, five main seeds, four tuning candidates.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nimport multiprocessing\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport ai4bci_study as b\n\n\nclass DepthwiseLIF(nn.Module):\n    """One LIF block; depthwise electrode filters and four temporal-bin features."""\n    def __init__(self, cfg):\n        super().__init__()\n        self.temporal = nn.Conv2d(1, 8, (1, 63), padding=(0, 31), bias=False)\n        self.spatial = nn.Conv2d(8, 16, (22, 1), groups=8, bias=True)\n        self.norm = nn.GroupNorm(1, 16)\n        self.head = nn.Linear(64, 4)\n        self.leak = cfg.leak\n\n    def forward(self, x, mode=0, threshold=1., features=False):\n        # Linear spatial/temporal filters commute; spatial first avoids 22-channel intermediates.\n        mixed = F.conv1d(x, self.spatial.weight.reshape(16, 22, 1))\n        kernels = self.temporal.weight.squeeze(2).repeat_interleave(2, dim=0)\n        drive = F.conv1d(mixed, kernels, padding=31, groups=16) + self.spatial.bias[None, :, None]\n        drive = self.norm(F.avg_pool1d(drive, 8))\n        spikes = b.lif_sequence(drive, self.leak, threshold, mode)\n        pooled = F.adaptive_avg_pool1d(spikes, 4).flatten(1)\n        logits = self.head(pooled)\n        return (logits, [spikes.mean(2), pooled]) if features else logits\n\n\ndef make_model(cfg, method, subject, seed, device):\n    if method != "ordinary":\n        raise ValueError("This replication tests readout recovery under ordinary training only")\n    b.seed_all(subject * 10000 + seed)\n    return DepthwiseLIF(cfg).to(device)\n\n\ndef configure_engine():\n    # Give resumes a distinct identity without changing the archived study code.\n    b.ENGINE_HASH = hashlib.sha256(\n        (Path(b.__file__).read_bytes() + Path(__file__).read_bytes())).hexdigest()\n    b.make_model = make_model\n\n\ndef protocol(cfg):\n    return dict(\n        analysis="Post-study exploratory replication; design fixed before its target evaluation",\n        architecture="single-block depthwise LIF; temporal kernel 63; 8 temporal filters; "\n                     "16 depthwise spatial filters; average pool 8; GroupNorm(1); "\n                     "LIF leak .9, threshold 1; four temporal-bin averages; linear 64-to-4 head",\n        comparison="Original smooth and raw-spike heads versus separately source-fitted ridge heads",\n        training="Ordinary AdamW only; original source-validation optimizer grid and checkpoint rule",\n        selection="All participant training and source-only ridge selection finish before target loading",\n        config=asdict(cfg), original_engine_sha=b.sha_file(b.__file__),\n        replication_sha=b.sha_file(__file__), combined_engine_sha=b.ENGINE_HASH,\n        software=b.environment(), independent_dataset=False,\n        limitations="Same nine participants and dataset; no replication of SAST superiority; "\n                    "offline preprocessing and normalization; not preregistered")\n\n\ndef train_source(cfg, subject, cache, root, device, threads):\n    configure_engine()\n    torch.set_num_threads(threads)\n    device = torch.device(device)\n    data = b.prepare_subject(cfg, subject, cache, root, target=False)\n    assert "test" not in data\n    directory = root / f"subject_{subject:02d}" / "ordinary"\n    candidates = []\n    for j, settings in enumerate(b.candidate_settings(cfg, "ordinary")):\n        print(f"SOURCE participant {subject}: pilot {j}", flush=True)\n        fitted = b.fit_model(cfg, "ordinary", subject, cfg.pilot_seed, settings,\n                             data, directory / f"pilot_{j}", device)\n        candidates.append(dict(candidate=j, settings=settings, validation_ba=fitted[1]["ba"],\n                               validation_loss=fitted[1]["loss"]))\n    winner = min(candidates, key=lambda c: (-c["validation_ba"], c["validation_loss"], c["candidate"]))\n    b.atomic_json(directory / "selection.json", dict(winner=winner, candidates=candidates))\n    xt, ft = data["train"]\n    xv, fv = data["val"]\n    for seed in cfg.seeds:\n        print(f"SOURCE participant {subject}: main seed {seed}", flush=True)\n        folder = directory / f"seed_{seed}"\n        model = b.fit_model(cfg, "ordinary", subject, seed, winner["settings"],\n                            data, folder, device)[0]\n        before = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        heads, choices = {}, {}\n        for mode, flag in (("smooth", 0), ("raw", 1)):\n            _, train_features = b.infer(model, xt, device, flag)\n            _, val_features = b.infer(model, xv, device, flag)\n            head = b.select_ridge(train_features[1], ft.label.to_numpy(),\n                                  val_features[1], fv.label.to_numpy(), cfg.ridge_alphas)\n            choices[mode] = dict(alpha=head["alpha"], source_validation_ba=b.balanced_accuracy(\n                fv.label, b.ridge_predict(head, val_features[1])))\n            heads.update({f"{mode}_{key}": value for key, value in head.items()})\n        np.savez_compressed(folder / "source_heads.npz", **heads)\n        b.atomic_json(folder / "source_readout_selection.json", choices)\n        assert all(torch.equal(before[k], v.cpu()) for k, v in model.state_dict().items())\n\n\ndef run(cfg, cache, output, device="cpu", workers=1, threads=2):\n    cfg.validate()\n    if workers < 1 or threads < 1:\n        raise ValueError("workers and threads must be positive")\n    configure_engine()\n    device = torch.device(device)\n    identity = protocol(cfg)\n    name = ("SMOKE_" if cfg.synthetic else "DEPTHWISE_LIF_") + b.digest(identity)[:16]\n    root = Path(output) / name\n    root.mkdir(parents=True, exist_ok=True)\n    b.atomic_json(root / "protocol.json", identity)\n    (root / "ai4bci_study.py").write_bytes(Path(b.__file__).read_bytes())\n    (root / "replicate_readout.py").write_bytes(Path(__file__).read_bytes())\n    workers = min(workers, len(cfg.subjects))\n    if device.type != "cpu" and workers != 1:\n        raise ValueError("Use one source-training worker on GPU")\n    if workers == 1:\n        for subject in cfg.subjects:\n            train_source(cfg, subject, cache, root, str(device), threads)\n    else:\n        with ProcessPoolExecutor(max_workers=workers, mp_context=multiprocessing.get_context("spawn")) as pool:\n            jobs = [pool.submit(train_source, cfg, subject, cache, root, str(device), threads)\n                    for subject in cfg.subjects]\n            for job in jobs:\n                job.result()\n    locked = {str(p.relative_to(root)): b.sha_file(p) for p in sorted(root.rglob("*"))\n              if p.name in {"best.pt", "selection.json", "source_heads.npz", "source_readout_selection.json"}}\n    b.atomic_json(root / "source_locked.json", locked)\n    rows = []\n    for subject in cfg.subjects:\n        data = b.prepare_subject(cfg, subject, cache, root, target=True)\n        xe, fe = data["test"]\n        for seed in cfg.seeds:\n            folder = root / f"subject_{subject:02d}" / "ordinary" / f"seed_{seed}"\n            model = make_model(cfg, "ordinary", subject, seed, device)\n            model.load_state_dict(torch.load(folder / "best.pt", map_location=device, weights_only=True))\n            saved = np.load(folder / "source_heads.npz", allow_pickle=False)\n            predictions = fe.copy()\n            for mode, flag in (("smooth", 0), ("raw", 1)):\n                logits, features = b.infer(model, xe, device, flag)\n                head = {k: saved[f"{mode}_{k}"] for k in ("weights", "intercept", "mean", "scale", "alpha")}\n                for kind, pred in (("original", logits.argmax(1)),\n                                   ("ridge", b.ridge_predict(head, features[1]))):\n                    key = f"{mode}_{kind}"\n                    predictions[key] = pred\n                    clean = fe.artifact.to_numpy() == 0\n                    rows.append(dict(subject=subject, seed=seed, mode=mode, readout=kind,\n                                     ba=b.balanced_accuracy(fe.label, pred),\n                                     artifact_free_ba=b.balanced_accuracy(fe.label[clean], pred[clean])))\n            predictions.to_csv(folder / "test_predictions.csv", index=False)\n    for path, sha in locked.items():\n        assert b.sha_file(root / path) == sha, f"Source fit changed during target evaluation: {path}"\n    scores = pd.DataFrame(rows)\n    scores.to_csv(root / "per_seed.csv", index=False)\n    participants = scores.groupby(["subject", "mode", "readout"]).ba.mean().unstack(["mode", "readout"])\n    estimates = []\n    def estimate(name, values):\n        mean, low, high = b.interval(values, cfg)\n        return dict(comparison=name, mean=mean, low=low, high=high)\n    for key in participants:\n        estimates.append(estimate("_".join(key), participants[key]))\n    contrasts = {\n        "raw_ridge_minus_raw_original": participants["raw", "ridge"] - participants["raw", "original"],\n        "smooth_ridge_minus_smooth_original": participants["smooth", "ridge"] - participants["smooth", "original"],\n        "raw_ridge_minus_smooth_ridge": participants["raw", "ridge"] - participants["smooth", "ridge"],\n        "excess_raw_recovery_over_smooth_recovery":\n            (participants["raw", "ridge"] - participants["raw", "original"])\n            - (participants["smooth", "ridge"] - participants["smooth", "original"])}\n    for name, values in contrasts.items():\n        estimates.append(estimate(name, values))\n    pd.DataFrame(estimates).to_csv(root / "summary.csv", index=False)\n    participants.columns = ["_".join(c) for c in participants.columns]\n    participants.to_csv(root / "participant_scores.csv")\n    b.atomic_json(root / "complete.json", dict(participants=len(cfg.subjects), seeds=len(cfg.seeds),\n                                               synthetic=cfg.synthetic, source_lock_verified=True,\n                                               results_rows=len(scores)))\n    print(root, flush=True)\n    print(pd.DataFrame(estimates).to_string(index=False), flush=True)\n    return root\n\n\ndef self_check():\n    cfg = replace(b.BCIConfig(), samples=64, synthetic=True)\n    model = DepthwiseLIF(cfg)\n    x = torch.randn(8, 22, 64)\n    mixed = F.conv1d(x, model.spatial.weight.reshape(16, 22, 1))\n    fast = F.conv1d(mixed, model.temporal.weight.squeeze(2).repeat_interleave(2, dim=0),\n                    padding=31, groups=16) + model.spatial.bias[None, :, None]\n    reference = model.spatial(model.temporal(x.unsqueeze(1))).squeeze(2)\n    torch.testing.assert_close(fast, reference, atol=2e-6, rtol=2e-5)\n    filters = (model.temporal.weight, model.spatial.weight, model.spatial.bias)\n    fast_grads = torch.autograd.grad(fast.square().mean(), filters)\n    reference_grads = torch.autograd.grad(reference.square().mean(), filters)\n    for actual, expected in zip(fast_grads, reference_grads):\n        torch.testing.assert_close(actual, expected, atol=2e-6, rtol=2e-5)\n    logits, features = model(x, features=True)\n    assert logits.shape == (8, 4) and features[1].shape == (8, 64)\n    logits.sum().backward()\n    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())\n    model.eval()\n    with torch.no_grad():\n        raw, fs = model(x, mode=1, features=True)\n        assert torch.all((fs[1] >= 0) & (fs[1] <= 1))\n        assert torch.allclose(raw, model.head(fs[1]))\n        assert torch.allclose(raw, torch.cat([model(t[None], mode=1) for t in x]), atol=2e-6)\n    print("Second-architecture shape, gradient, binary-feature and rebatching checks passed")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--cache", default="replication/raw_data")\n    parser.add_argument("--output", default="replication/runs")\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--threads", type=int, default=2)\n    parser.add_argument("--workers", type=int, default=None)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    torch.set_num_threads(args.threads)\n    self_check()\n    cfg = replace(b.smoke_config(), subjects=(1,), widths=(16, 32)) if args.smoke else b.BCIConfig()\n    workers = args.workers if args.workers is not None else (3 if args.device == "cpu" else 1)\n    run(cfg, args.cache, args.output, args.device, workers=workers, threads=args.threads)\n'}
EXPECTED_SHA256 = {'ai4bci_study.py': '42ba021e26724e3e0d18ec77d1df765cc438092cf487c2eb1b94b8d5a56aacde', 'replicate_readout.py': '400b8140b071ce9819ae39e1f0b3c390810be0e9222932a5d952f7fa16b6f24b'}

ENGINE_DIR = Path(tempfile.mkdtemp(prefix="ai4bci_replication_engine_"))
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED_SHA256[name]
    (ENGINE_DIR / name).write_text(source)
print("Embedded source hashes:", EXPECTED_SHA256)

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
workers = 1 if device == "cuda" else min(3, os.cpu_count() or 1)
command = [sys.executable, "-u", str(ENGINE_DIR / "replicate_readout.py"),
           "--cache", str(PROJECT / "raw_data"), "--output", str(PROJECT / "runs"),
           "--device", device, "--workers", str(workers)]
subprocess.run(command + ["--smoke"], check=True)
print("Synthetic check passed; these are not paper results.")

In [ ]:
if RUN_FULL_REPLICATION:
    subprocess.run(command, check=True)
else:
    print("Full real-data replication disabled; no real replication result is available.")

In [ ]:
import zipfile
from IPython.display import FileLink, display
roots = sorted((PROJECT / "runs").glob("DEPTHWISE_LIF_*/complete.json"),
               key=lambda p: p.stat().st_mtime)
if RUN_FULL_REPLICATION and roots:
    ROOT = roots[-1].parent
    import pandas as pd
    display(pd.read_csv(ROOT / "summary.csv"))
    ARCHIVE = PROJECT / (ROOT.name + "_results.zip")
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob("*")):
            if p.is_file() and p.name != "resume.pt" and not p.name.startswith("epochs_"):
                z.write(p, p.relative_to(ROOT))
    display(FileLink(str(ARCHIVE)))
    print("Results:", ROOT, "Download:", ARCHIVE)